# 02 — Train Qwen3-8B with QLoRA

The second of three notebooks (01 tests the base model, **02 trains**, 03 tests the trained model). It trains a small LoRA adapter on the training examples and saves it to Drive — no answers are generated here, so it uses the GPU time for training only.

**Before you start:** *Runtime → Change runtime type → **T4 GPU***, then run the steps in order.

**If Colab disconnects during training:** start a new session and run every step again. Step 8 finds the last saved epoch in `checkpoints/` on Drive and **continues from it** — you lose at most the epoch that was running. (This needs `USE_DRIVE = True`.)

## Step 1 — Check the GPU
Confirms a GPU is attached and which one. On a T4 you should see **Tesla T4, compute capability 7.5**. The T4 has no real bfloat16 support, so float16 is used on it (bfloat16 only on newer GPUs such as A100/L4).

In [ ]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
major, minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)}  |  compute capability {major}.{minor}")
print("Precision used:", "bfloat16" if major >= 8 else "float16 (this GPU has no real bfloat16)")

## Step 2 — Install the libraries
- **transformers** — loads Qwen3 (pinned to 4.x, 4.56 or newer — this code uses the 4.x API).
- **peft** — the LoRA adapter. **bitsandbytes** — 4-bit loading (the "Q" in QLoRA).
- **accelerate**, **datasets** — GPU placement and the data container.

Takes 1–2 minutes. If Colab asks you to **restart the session**, do it, then continue with Step 3.

In [ ]:
!pip install -q "transformers>=4.56,<5.0" "peft>=0.13" "bitsandbytes>=0.43" "accelerate>=0.34" datasets
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

## Step 3 — Get the data and choose where results go
Downloads the repository (the data is in its `data/` folder) and sets the folders. **Keep these settings the same in all three notebooks** — notebook 03 finds the adapter from 02 and the base results from 01 through them.
- **`DATA_VERSION`** — which data: `"v2"` uses `questions_v2_train.jsonl` / `questions_v2_val.jsonl`.
- **`USE_DRIVE = True`** *(recommended)* — outputs go to `MyDrive/kqml_finetune/` and survive a disconnect. With `False` they stay on Colab's disk and are lost when the session ends.

Every output of a version goes in its own folder, `qwen3-8b-<version>`, so v1 and v2 never mix.

In [ ]:
from pathlib import Path

DATA_VERSION = "v2"                  # questions_<version>_train.jsonl / _val.jsonl
USE_DRIVE    = True                  # False = Colab disk only (lost when the session ends)
REPO         = "https://github.com/ishancoderr/kqml-geo-qwen3-qlora.git"
BRANCH       = "main"

!rm -rf /content/kqml-geo-qwen3-qlora
!git clone -q --depth 1 --branch {BRANCH} {REPO} /content/kqml-geo-qwen3-qlora
DATA_DIR   = Path("/content/kqml-geo-qwen3-qlora/data")
TRAIN_FILE = DATA_DIR / f"questions_{DATA_VERSION}_train.jsonl"
VAL_FILE   = DATA_DIR / f"questions_{DATA_VERSION}_val.jsonl"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/kqml_finetune")
else:
    WORK_DIR = Path("/content/kqml_finetune")

RUN_NAME       = f"qwen3-8b-{DATA_VERSION}"
ADAPTER_DIR    = WORK_DIR / "adapters" / RUN_NAME       # written by 02, read by 03
CHECKPOINT_DIR = WORK_DIR / "checkpoints" / RUN_NAME    # 02 only: one checkpoint per epoch
RESULTS_DIR    = WORK_DIR / "results" / RUN_NAME        # eval_base.jsonl, eval_tuned.jsonl, training log
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for f in (TRAIN_FILE, VAL_FILE):
    print(("found    " if f.exists() else "MISSING  ") + str(f))
print("outputs go to:", WORK_DIR, "| run:", RUN_NAME)

## Step 4 — Settings
- **MAX_LENGTH 2048** — longest example allowed (v2's longest is about 1,830 tokens, so none is skipped).
- **EPOCHS 3** — passes over the training data. The best epoch (lowest validation loss) is kept.
- **LEARNING_RATE 2e-4**, **LORA_RANK 16** — standard QLoRA values.
- **BATCH_SIZE 1 × GRAD_ACCUM 8** — one example at a time on the T4; gradients from 8 are added before each update (an effective batch of 8).

In [ ]:
import torch

MODEL_NAME    = "Qwen/Qwen3-8B"
MAX_LENGTH    = 2048
EPOCHS        = 3
LEARNING_RATE = 2e-4
LORA_RANK     = 16
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
SEED          = 42

DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("compute dtype:", DTYPE)
for d in (ADAPTER_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Step 5 — Read the data
Each line of the JSONL is **one LLM call** of the agent: system prompt, user message, the correct answer, and a `meta` block. From `meta` we keep the **task** (classify / extract / sql), the question id and the category, so the scores can be shown per task and per category.

In [ ]:
import json
from collections import Counter

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            msgs = rec["messages"]
            assert [m["role"] for m in msgs] == ["system", "user", "assistant"], msgs
            meta = rec.get("meta", {})
            rows.append({"task": meta.get("task", "all"), "qid": meta.get("qid", ""),
                         "category": meta.get("category", ""),
                         "prompt_messages": msgs[:2], "answer": msgs[2]["content"]})
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows   = load_jsonl(VAL_FILE)
print("train:", len(train_rows), dict(Counter(r["task"] for r in train_rows)))
print("val:  ", len(val_rows),   dict(Counter(r["task"] for r in val_rows)))

## Step 6 — Tokenizer: turn each example into training data
1. The system + user messages are rendered with **Qwen3's chat template, thinking off** — the prompt the model gets when the agent uses it.
2. The **target** is the correct answer followed by `<|im_end|>`, so the model also learns where to stop.
3. Every prompt token gets the label `-100` — **no loss there**: the model learns to produce the answer, not to repeat the prompt.
4. An example longer than MAX_LENGTH is **skipped, never cut short**. Check that the output says *skipped 0* everywhere.

In [ ]:
from transformers import AutoTokenizer

END = "<|im_end|>"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_text(row):
    return tokenizer.apply_chat_template(row["prompt_messages"], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)

def encode(row):
    prompt_ids = tokenizer(prompt_text(row), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(row["answer"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}

def encode_all(rows, name):
    kept, skipped, lengths = [], Counter(), {}
    for r in rows:
        item = encode(r)
        lengths.setdefault(r["task"], []).append(len(item["input_ids"]))
        if len(item["input_ids"]) > MAX_LENGTH:
            skipped[r["task"]] += 1
            continue
        kept.append(item)
    print(f"{name}: {len(kept)}/{len(rows)} kept")
    for t, ls in sorted(lengths.items()):
        print(f"   {t:<9} n={len(ls):<4} tokens mean {sum(ls) // len(ls):<5} max {max(ls):<5} skipped {skipped[t]}")
    return kept

train_data = encode_all(train_rows, "train")
val_data   = encode_all(val_rows, "val")

sample = val_rows[0]
item = encode(sample)
print("\n--- end of the prompt the model sees ---\n..." + prompt_text(sample)[-200:])
print("--- the only part the loss is computed on ---")
print(tokenizer.decode([t for t, l in zip(item["input_ids"], item["labels"]) if l != -100]))

## Step 7 — Load Qwen3-8B in 4-bit
Downloads the model (about **16 GB**, a few minutes, once per session) and loads it **quantised to 4-bit NF4**, about 6 GB of GPU memory. Generation is switched to plain greedy decoding, so every answer is deterministic.

If the session **crashes while loading**, it ran out of *system RAM* (free Colab has about 12 GB), not GPU memory — restart the session and run again from Step 1.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
    device_map={"": 0},
    dtype=DTYPE,
)
gen = model.generation_config
gen.do_sample, gen.temperature, gen.top_p, gen.top_k = False, None, None, None
print(f"loaded — GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Step 8 — Add LoRA and train
1. **Prepare the 4-bit model for training** — gradient checkpointing (recompute instead of store, to save memory).
2. **Add LoRA**: rank-16 matrices on all seven linear layers of every block. The 8B base weights stay frozen; about 0.5% of the parameters train.
3. **Train** — training loss every 5 steps, **validation loss** after each epoch.
4. A checkpoint is saved to Drive after every epoch; at the end the **epoch with the lowest validation loss** is kept.

If training loss keeps falling while validation loss rises, the model is memorising — that is why the best epoch, not the last, is kept.

**Re-running this step in the same session:** run Step 7 first, so LoRA is not added twice.

In [ ]:
import time
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, set_seed

if isinstance(model, PeftModel):
    raise RuntimeError("LoRA is already attached — run Step 7 again before re-running this step.")

set_seed(SEED)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "labels":         torch.tensor([b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
    }

use_bf16 = DTYPE == torch.bfloat16
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        optim="paged_adamw_8bit",
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    ),
    train_dataset=Dataset.from_list(train_data),
    eval_dataset=Dataset.from_list(val_data),
    data_collator=collate,
)

resume = any(CHECKPOINT_DIR.glob("checkpoint-*"))
print("resuming from the last checkpoint" if resume else "starting fresh")
t0 = time.time()
trainer.train(resume_from_checkpoint=True if resume else None)
print(f"\ntraining took {(time.time() - t0) / 60:.1f} min")
for row in trainer.state.log_history:
    if "eval_loss" in row:
        print(f"  epoch {row['epoch']:.0f}: validation loss {row['eval_loss']:.4f}")

## Step 9 — Save the adapter and the training log
Saves only the trained LoRA weights (about **175 MB**) to `adapters/qwen3-8b-<version>/` — notebook 03 loads them from there. The loss history goes to `results/…/training_log.json`, for the loss curve in the thesis.

In [ ]:
trainer.save_model(str(ADAPTER_DIR))        # the best epoch, by validation loss
tokenizer.save_pretrained(str(ADAPTER_DIR))
log_path = RESULTS_DIR / "training_log.json"
log_path.write_text(json.dumps({"best_checkpoint": trainer.state.best_model_checkpoint,
                                "log_history": trainer.state.log_history}, indent=1), encoding="utf-8")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size / 1e6:8.1f} MB")
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("log saved:", log_path)

## Step 10 — Download the adapter (only needed when `USE_DRIVE = False`)
With Drive the adapter is already in `MyDrive/kqml_finetune/adapters/`. Checkpoints are left out — large, and only needed to resume training.

In [ ]:
import shutil
from google.colab import files

bundle = Path("/content/bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(ADAPTER_DIR, bundle / "adapter")
shutil.copytree(RESULTS_DIR, bundle / "results")
zip_path = shutil.make_archive(f"/content/{RUN_NAME}-adapter", "zip", bundle)
print(f"{Path(zip_path).name}: {Path(zip_path).stat().st_size / 1e6:.0f} MB")
files.download(zip_path)